In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, to_date, max as _max,
    coalesce, lit, upper, trim, abs
)
from delta.tables import DeltaTable

# --------------------------------------------------
# Configuration
# --------------------------------------------------
bronze_table = "bronze_dev.global_mart_retail.raw_data"
fact_table   = "silver_dev.global_mart_retail.fact_sales"

dim_customer = "silver_dev.global_mart_retail.dim_customer"
dim_product  = "silver_dev.global_mart_retail.dim_product"
dim_date     = "silver_dev.global_mart_retail.dim_date"

# --------------------------------------------------
# Determine last loaded ingestion_ts (Incremental)
# --------------------------------------------------
if spark.catalog.tableExists(fact_table):
    last_loaded_ts = (
        spark.table(fact_table)
        .agg(_max("ingestion_ts").alias("max_ingestion_ts"))
        .collect()[0]["max_ingestion_ts"]
    )
else:
    last_loaded_ts = None

# --------------------------------------------------
# Load Bronze Data Incrementally
# --------------------------------------------------
bronze_df = spark.table(bronze_table)

if last_loaded_ts:
    bronze_df = bronze_df.filter(col("ingestion_ts") > last_loaded_ts)

# --------------------------------------------------
# Basic Data Quality & Standardization (Silver rules)
# --------------------------------------------------
bronze_df = (
    bronze_df
    # Standardize IDs before joins
    .withColumn("customer_id", upper(trim(col("customer_id"))))
    .withColumn("product_id", upper(trim(col("product_id"))))

    # Returns handling (keep records, flag implicitly via negative quantity)
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("sales", col("sales").cast("decimal(12,2)"))
    .withColumn("discount", col("discount").cast("decimal(12,2)"))
    .withColumn("profit", col("profit").cast("decimal(12,2)"))
    
    # Flag returns explicitly
    .withColumn("is_return", col("quantity") < 0)
)

# --------------------------------------------------
# Join Dimensions (Current Records Only)
# --------------------------------------------------
fact_df = (
    bronze_df.alias("b")
    .join(
        spark.table(dim_customer).alias("c")
        .filter(col("is_current_record") == True),
        col("b.customer_id") == upper(col("c.customer_id")),
        "left"
    )
    .join(
        spark.table(dim_product).alias("p")
        .filter(col("is_current_record") == True),
        col("b.product_id") == upper(col("p.product_id")),
        "left"
    )
    .join(
        spark.table(dim_date).alias("od"),
        to_date(col("b.order_date")) == col("od.date"),
        "left"
    )
    .join(
        spark.table(dim_date).alias("sd"),
        to_date(col("b.ship_date")) == col("sd.date"),
        "left"
    )
    .select(
        # ---- SURROGATE KEYS ----
        coalesce(col("c.customer_key"), lit(-1)).alias("customer_key"),
        coalesce(col("p.product_key"), lit(-1)).alias("product_key"),
        coalesce(col("od.date_key"), lit(19500101)).alias("order_date_key"),
        coalesce(col("sd.date_key"), lit(19500101)).alias("ship_date_key"),

        # ---- DEGENERATE DIMENSIONS ----
        col("b.order_id").alias("order_id"),
        col("b.ship_mode").alias("ship_mode"),

        # ---- MEASURES (Raw, Gold will derive KPIs) ----
        col("b.quantity").alias("order_quantity"),
        col("b.sales").alias("sales_amount"),
        col("b.discount").alias("discount_amount"),
        col("b.profit").alias("profit_amount"),

        # ---- Returns flag ----
    	(col("b.quantity") < 0).alias("is_returned"),
    
    	# Optional (future-friendly, not KPI)
        abs(col("b.quantity")).alias("abs_order_quantity"),
        abs(col("b.sales")).alias("abs_sales_amount"),

        # ---- AUDIT ----
        col("b.ingestion_ts").cast("timestamp"),
        col("b.source_file_path").cast("string"),
        current_timestamp().cast("timestamp").alias("load_timestamp")
    )
)

# --------------------------------------------------
# Record Counts (Audit)
# --------------------------------------------------
before_count = (
    spark.table(fact_table).count()
    if spark.catalog.tableExists(fact_table)
    else 0
)

# --------------------------------------------------
# Write to Silver Fact (Append Only)
# --------------------------------------------------
fact_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(fact_table)

after_count = spark.table(fact_table).count()
inserted_records = after_count - before_count

# --------------------------------------------------
# Logging
# --------------------------------------------------
print("Fact Sales Load Completed")
print(f"Records inserted in this run : {inserted_records}")
print(f"Total records in fact table : {after_count}")
